# Streaming from multiple files

In [5]:
from pathlib import Path

PATH_DATADIR= Path("stream")

In [21]:
from typing import Callable

def read_data(filepath: Path):
    print(f"opening file {str(filepath)}")
    with open(filepath, "r") as f:
        return [*map(int, f.readlines())]

def get_stream(datafiles: list[Path], transform: Callable | None=None) -> Callable:
    in_memory = []
    datafiles = list(datafiles)
    def helper(batch_size: int):
        nonlocal in_memory
        while len(in_memory) < batch_size:
            if not datafiles:
                last_one = in_memory[:]
                in_memory = []
                return last_one
            data = read_data(datafiles.pop(0))
            in_memory.extend(transform(data) if transform is not None else data)
        batch = in_memory[:batch_size]
        in_memory = in_memory[batch_size:]
        return batch
    return helper

stream = get_stream(
    [f for f in PATH_DATADIR.iterdir() if f.name.startswith("data")],
    transform = lambda xs: [x * 2 for x in xs]
)

In [23]:
stream(10)

opening file stream/data2.txt


[22, 24, 26, 28, 30, 32]

In [ ]:
def make_test_stream(chunks, transform=None):
    """Mirror of get_stream using in-memory lists instead of files."""
    in_memory = []
    chunks = [list(c) for c in chunks]
    def helper(batch_size):
        nonlocal in_memory
        while len(in_memory) < batch_size:
            if not chunks:
                last = in_memory[:]
                in_memory = []
                return last
            data = chunks.pop(0)
            in_memory.extend(transform(data) if transform is not None else data)
        batch = in_memory[:batch_size]
        in_memory = in_memory[batch_size:]
        return batch
    return helper

# normal: two equal-size chunks, batch divides evenly
s = make_test_stream([[1,2,3],[4,5,6]]); assert s(2)==[1,2] and s(2)==[3,4] and s(2)==[5,6] and s(2)==[]
# batch crosses a file boundary
s = make_test_stream([[1,2,3],[4,5,6]]); assert s(4)==[1,2,3,4] and s(4)==[5,6] and s(4)==[]
# batch_size larger than all data: returns everything at once, then []
s = make_test_stream([[1,2]]); assert s(5)==[1,2] and s(5)==[]
# single-item batches
s = make_test_stream([[1,2,3]]); assert s(1)==[1] and s(1)==[2] and s(1)==[3] and s(1)==[]
# batch_size matches chunk size exactly: one file per call
s = make_test_stream([[1,2,3],[4,5,6]]); assert s(3)==[1,2,3] and s(3)==[4,5,6] and s(3)==[]
# repeated calls after exhaustion keep returning []
s = make_test_stream([[1]]); s(10); assert s(3)==[] and s(3)==[]
# empty chunk list
s = make_test_stream([]); assert s(3)==[]
# single chunk, single item
s = make_test_stream([[42]]); assert s(1)==[42] and s(1)==[]
# transform=None is identical to identity
s = make_test_stream([[1,2,3],[4,5,6]], transform=None); assert s(4)==[1,2,3,4] and s(4)==[5,6] and s(4)==[]
# transform is applied per chunk before buffering
s = make_test_stream([[1,2,3],[4,5,6]], transform=lambda xs: [x*2 for x in xs]); assert s(4)==[2,4,6,8] and s(4)==[10,12] and s(4)==[]
# transform can filter (return fewer items than input)
s = make_test_stream([[1,2,3,4],[5,6,7,8]], transform=lambda xs: [x for x in xs if x % 2 == 0]); assert s(2)==[2,4] and s(2)==[6,8] and s(2)==[]
# transform can return empty list (chunk filtered out entirely)
s = make_test_stream([[1,3,5],[2,4,6]], transform=lambda xs: [x for x in xs if x % 2 == 0]); assert s(2)==[2,4] and s(2)==[6] and s(2)==[]

print("all assertions passed")